# Colab: bellman run and EdDSA test (phase decomposition, native mode)
Runs, one after the other on the **same** Colab machine: the 21 circuits with the bellman back end, then the two EdDSA verifiers with ark, then with bellman.
Run the cells **from the top, one at a time** (Shift+Enter). Runtime > Change runtime type must be **CPU**. Keep this tab open while it runs (cell 5). Expected time: about 3 to 4 hours.

In [ ]:
# 1. Upload TWO files when the button appears: zkp_v4d.zip and zkp_colab_runs_kit.zip
import os, zipfile, shutil, glob
from google.colab import files
up = files.upload()
for f in up:
    if f.endswith('.zip'): zipfile.ZipFile(f).extractall('/content')
K = '/content/zkp_colab_runs'
for f in glob.glob(K + '/*'):
    if os.path.isfile(f): shutil.copy(f, '/content/zkp_v4/')
for f in glob.glob(K + '/circuits/*'): shutil.copy(f, '/content/zkp_v4/circuits/')
os.chdir('/content/zkp_v4')
!chmod +x zokrates_timers zok_wrap.sh; sha256sum zokrates_timers; ls circuits | grep eddsa

In [ ]:
# 2. Connect Google Drive: every finished pass is copied there, so an interrupted session continues from that copy
from google.colab import drive
drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/zkp_colab_runs'
os.makedirs(BACKUP, exist_ok=True)
!cp -rn "$BACKUP"/phases_* /content/zkp_v4/ 2>/dev/null; ls -d phases_* 2>/dev/null

In [ ]:
# 3. Three-minute check: it must print READY, then two EdDSA lines with bellman, and no line may start with !!!
!bash check_setup_linux.sh | tail -3 && bash run_queue_linux.sh "bellman:eddsaquick" 2>&1 | tail -6

In [ ]:
# 4. Start the three runs in the background (an unfinished run continues by itself; finished ones are skipped)
RUNS = "bellman:full ark:eddsa bellman:eddsa"
os.environ['AFTER_PASS_HOOK'] = f'cp -r "$RES" "{BACKUP}/"; true'
!nohup bash run_queue_linux.sh "{RUNS}" > run_queue.out 2>&1 &

In [ ]:
# 5. Watch it (prints every 5 minutes; keep this tab open until it says QUEUE FINISHED)
import subprocess, time
from IPython.display import clear_output
def running(): return subprocess.run(['pgrep', '-f', 'run_queue_linux.sh'], capture_output=True).returncode == 0
time.sleep(20)
while running():
    out = subprocess.run("grep -a '^===' run_queue.out | tail -2; grep -a '^\\[' run_queue.out | tail -2; tail -1 run_queue.out", shell=True, capture_output=True, text=True).stdout
    clear_output(wait=True); print(time.strftime('%H:%M:%S'), '\n' + out)
    time.sleep(300)
print(subprocess.run("grep -a '^===' run_queue.out; tail -15 run_queue.out", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 6. Pack all finished result folders into one zip and download it (a copy also goes to Drive)
ds = [d for d in sorted(glob.glob('phases_*')) if os.path.isdir(d) and not d.endswith('_quick')]
print(ds)
z = 'zkp_colab_runs_results.zip'
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
    for d in ds:
        for root, _, fs in os.walk(d):
            for f in fs: zf.write(os.path.join(root, f))
!cp "{z}" "$BACKUP"/
files.download(z)